# Project 01: Exploratory Data Analysis (EDA) on the Titanic Dataset

### Guiding Questions
1. Did women survive at a significantly higher rate than men?
2. How strongly did ticket class (`pclass`) influence passenger survival?
3. Was age related to survival across different life stages (e.g., children vs. adults)?

In [1]:
import pandas as pd
import seaborn as sns
df = sns.load_dataset("titanic")
print("Dataset Shape:", df.shape)
display(df.head())
print("\nDataset Info:")
df.info()
print("\nSummary Statistics:")
display(df.describe())

Dataset Shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True



Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB

Summary Statistics:


,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


### survived: Target variable (0 = No, 1 = Yes).

### pclass: Ticket class (1 = 1st/Upper, 2 = 2nd/Middle, 3 = 3rd/Lower).

### sex: Passenger gender (male, female).

### age: Passenger age in years.

### sibsp / parch: Number of siblings/spouses and parents/children aboard.

### fare: Ticket price paid.

### embarked / embark_town: Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton).

### deck: Cabin deck letter.

In [2]:
missing_counts = df.isna().sum().sort_values(ascending=False)
print("Missing Values per Column:\n", missing_counts[missing_counts > 0])

Missing Values per Column:
 deck           688
age            177
embarked         2
embark_town      2
dtype: int64


In [3]:
df["age"] = df["age"].fillna(
    df.groupby(["pclass", "sex"])["age"].transform("median")
)
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])
df = df.drop(columns=["deck"])
print("Remaining missing values:", df.isna().sum().sum())

Remaining missing values: 0


### Age: Imputed using the median age of each passenger's specific pclass and sex group rather than the global mean, because passenger age varied across socioeconomic classes (1st class passengers tended to be older than 3rd class passengers), and the median is resistant to extreme age outliers.
### Deck: Dropped completely because over 77% (688 out of 891 rows) of its values are missing, making imputation unreliable.
### Embarked / Embark Town: Filled with the mode (S / Southampton) because only 2 rows were missing, so assigning the most frequent port introduces negligible bias.

In [4]:
dup_count = df.duplicated().sum()
print(f"Duplicate rows found: {dup_count}")
df_clean = df.drop_duplicates().copy()
print(f"Shape after removing duplicates: {df_clean.shape}")
categorical_cols = ["sex", "embarked", "class", "who", "embark_town", "alive"]
for col in categorical_cols:
    df_clean[col] = df_clean[col].astype("category")
df_clean.info()

Duplicate rows found: 118
Shape after removing duplicates: (773, 14)
<class 'pandas.DataFrame'>
Index: 773 entries, 0 to 890
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     773 non-null    int64   
 1   pclass       773 non-null    int64   
 2   sex          773 non-null    category
 3   age          773 non-null    float64 
 4   sibsp        773 non-null    int64   
 5   parch        773 non-null    int64   
 6   fare         773 non-null    float64 
 7   embarked     773 non-null    category
 8   class        773 non-null    category
 9   who          773 non-null    category
 10  adult_male   773 non-null    bool    
 11  embark_town  773 non-null    category
 12  alive        773 non-null    category
 13  alone        773 non-null    bool    
dtypes: bool(2), category(6), float64(2), int64(4)
memory usage: 48.6 KB


### Seaborn Titanic dataset lacks unique passenger IDs or names, some rows with identical age, class, fare, and sex appear as duplicates (~107 rows).

In [5]:
sex_survival = (
    df.groupby("sex")["survived"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "survival_rate", "count": "passenger_count"})
    .sort_values(by="survival_rate", ascending=False)
    .round(2)
)
print("--- Survival Rate by Sex ---")
display(sex_survival)
class_survival = (
    df.groupby("pclass")["survived"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "survival_rate", "count": "passenger_count"})
    .sort_values(by="survival_rate", ascending=False)
    .round(2)
)
print("\n--- Survival Rate by Ticket Class ---")
display(class_survival)
df["age_band"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 40, 60, 100],
    labels=["Child", "Teen", "Adult", "Middle-aged", "Senior"]
)

age_survival = (
    df.groupby("age_band", observed=True)["survived"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "survival_rate", "count": "passenger_count"})
    .sort_values(by="survival_rate", ascending=False)
    .round(2)
)
print("\n--- Survival Rate by Age Band ---")
display(age_survival)

--- Survival Rate by Sex ---


,survival_rate,passenger_count
sex,,
female,0.74,314
male,0.19,577



--- Survival Rate by Ticket Class ---


,survival_rate,passenger_count
pclass,,
1,0.63,216
2,0.47,184
3,0.24,491



--- Survival Rate by Age Band ---


,survival_rate,passenger_count
age_band,,
Child,0.58,69
Teen,0.43,70
Middle-aged,0.39,128
Adult,0.36,602
Senior,0.23,22


### Key Findings & Summary

1. **Gender Disparity**: Female passengers had a significantly higher survival rate at **74%** (n = 314) compared to male passengers at **19%** (n = 577).
2. **Socioeconomic Class Association**: Higher ticket class was strongly associated with survival, with **63%** of 1st-class passengers surviving compared to **47%** in 2nd class and **24%** in 3rd class.
3. **Age Group Patterns**: Children (aged 0–12) achieved the highest survival rate among all age brackets at **58%**, whereas seniors (aged 60+) had the lowest survival rate at **23%**, while adults (18–40) survived at **36%**.